## Filesystem Pool: 100 Random PyTorch Tensors

A compact end-to-end example that creates 100 random PyTorch tensors, stores them in a `FilesystemPool`, reads a sample back, and verifies integrity.

### Notes
- `FilesystemPool` always mounts under `~/.laila/pools/.mnt/<pool_id>`.
- If that mountpoint is already mounted, the pool reuses it.
- If the image file already exists in the chosen image directory, the pool mounts it.
- If neither exists, the pool creates `<pool_id>.img` in the chosen image directory and mounts it.
- The user only provides the image directory; the image filename and mountpoint are fixed by the pool.


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append("/home/ubuntu/")

In [3]:
import os
import tempfile

import torch
import laila

/home/ubuntu/laila/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from laila.pool import FilesystemPool

temp_root = tempfile.mkdtemp(prefix="laila_fs_example_")

filesystem_pool = FilesystemPool(image_dir=temp_root)

laila.add_pool(filesystem_pool, pool_nickname="fs_pool")

print("Image dir :", filesystem_pool.image_dir)
print("Image path:", filesystem_pool.image_path)
print("Mount dir :", filesystem_pool.mount_dir)

Image path: /tmp/laila_fs_example_s84u72tw/filesystem_pool.img
Mount dir : /tmp/laila_fs_example_s84u72tw/mount


In [5]:
torch.manual_seed(42)

wrapped_tensors = []
for i in range(100):
    tensor = torch.randn(16, 16)
    wrapped = laila.constant(data=tensor, nickname=f"filesystem-random-tensor-{i}")
    wrapped_tensors.append(wrapped)

print("Created tensors:", len(wrapped_tensors))
print("First global_id:", wrapped_tensors[0].global_id)

Created tensors: 100
First global_id: LAILA:ENTRY:GLOBAL_ID:0ebe2807-5034-427c-a994-ca0bb284e19a


In [6]:
futures = []
for wrapped in wrapped_tensors:
    future = laila.memorize(wrapped, pool_nickname="fs_pool")
    futures.append(future)

for future in futures:
    future.wait()

stored_files = sorted(name for name in os.listdir(filesystem_pool.mount_dir) if name.endswith(".json"))
print("Stored tensors:", len(stored_files))
print("First stored file:", stored_files[0])

Stored tensors: 100
First stored file: LAILA%3AENTRY%3AGLOBAL_ID%3A00022938-1e61-46c7-93a2-5655c077579d.json


In [7]:
sample_indices = [0, 17, 42, 99]

for idx in sample_indices:
    original = wrapped_tensors[idx].data
    future = laila.remember(wrapped_tensors[idx].global_id, pool_nickname="fs_pool")
    future.wait()
    recovered = future.result
    assert torch.equal(recovered.data, original)

print("Verified sample tensors:", sample_indices)

Verified sample tensors: [0, 17, 42, 99]


In [8]:
for wrapped in wrapped_tensors:
    future = laila.forget(wrapped.global_id, pool_nickname="fs_pool")
    future.wait()

filesystem_pool.close()
print("Cleanup complete")

Cleanup complete
